# Orquestrador Comparativo — Flow Shop Scheduling

Notebook que reune todos os solvers dos alunos e os compara usando
as mesmas instancias, seeds e metricas.

**Solvers:** ACO (aluno_1) | Neuro-BOA (aluno_3) | BRKGA (aluno_4) | PBIL-Fuzzy (aluno_2)
**Engine:** `FlowShopEngine` oficial compartilhada
**Modo:** NPFS (Non-Permutation Flow Shop)
**Objetivo principal:** Makespan (Tardiness tambem e calculado)

## 0. Clonar o repositorio

Usa caminho absoluto para evitar duplicacao de diretorio ao rodar a celula mais de uma vez.

In [ ]:
REPO_URL = "https://github.com/EuRonald123/FlowShop_S.git"
DESTINO = "/content/FlowShop_S"

!rm -rf {DESTINO}
!git clone {REPO_URL} {DESTINO}

# Muda para o diretorio
%cd {DESTINO}

import os
ROOT = os.getcwd()
print(f"Diretorio de trabalho: {ROOT}")

## 1. Setup — Instalacao e Imports

In [ ]:
!pip install -q pymoo scikit-fuzzy optuna numpy matplotlib pandas scipy

# PyTorch (CPU basta para o Neuro-BOA)
import torch
if not torch.cuda.is_available():
    !pip install -q torch --index-url https://download.pytorch.org/whl/cpu

print("Dependencias instaladas.")

In [ ]:
# IMPORTS
import sys, time, copy, random
from pathlib import Path
from typing import List, Dict, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.style.use('default')

# Engine oficial
sys.path.insert(0, ROOT)
from src.flowshop_engine import FlowShopEngine

# Solver adapter
from src.solvers.adapter import (
    rodar_experimento,
    listar_solvers_disponiveis,
    SolverResult,
)

print("Imports concluidos.")
print(f"Solvers disponiveis: {listar_solvers_disponiveis()}")

## 2. Dados

Extrai as instancias do `Small.zip` se o diretorio `data/Small` estiver vazio.

In [ ]:
# DADOS: extrair Small.zip se necessario
DATA_DIR = Path(ROOT) / "data" / "Small"

if not DATA_DIR.exists() or len(list(DATA_DIR.glob("*.txt"))) == 0:
    zip_path = Path(ROOT) / "data" / "Small.zip"
    if zip_path.exists():
        import zipfile
        with zipfile.ZipFile(str(zip_path), "r") as z:
            z.extractall(str(Path(ROOT) / "data"))
        print(f"Instancias extraidas para {DATA_DIR}")
    else:
        print(f"AVISO: {DATA_DIR} vazio e sem Small.zip. Faca upload manual.")
else:
    print(f"Instancias encontradas: {len(list(DATA_DIR.glob('*.txt')))} arquivos")

## 3. Configuracao Geral

In [ ]:
# CONFIG GERAL

RESULTADOS_DIR = Path(ROOT) / "resultados_comparacao"
RESULTADOS_DIR.mkdir(parents=True, exist_ok=True)

# Instancias a testar (filtro por padrao no nome do arquivo)
# Formato do nome: I_{m}_{n}_{d}_{id}.txt
# Ex: I_2_16_5_1.txt = 2 maquinas, 16 jobs, grupo d=5, id=1
INSTANCIAS_CONFIG = [
    {"rotulo": "4x2",  "padrao": "I_2_4_",   "desc": "4 jobs, 2 maq. — smoke test"},
    {"rotulo": "16x2", "padrao": "I_2_16_2_", "desc": "16 jobs, 2 maq."},
    {"rotulo": "16x5", "padrao": "I_2_16_5_", "desc": "16 jobs, 5 maq. — instancia alvo (aluno_4)"},
    {"rotulo": "16x4", "padrao": "I_4_16_5_", "desc": "16 jobs, 4 maq."},
]

# Solvers e variantes a testar
SOLVERS_A_TESTAR = [
    "ACO",
    "BRKGA_fixo_default",
    "BRKGA_fuzzy",
    "Neuro-BOA_adaptativo",
    "PBIL-Fuzzy",
]

# Parametros de execucao
N_SEEDS = 5                     # sementes por solver x instancia
SEED_INICIO = 100                # seeds: range(INICIO, INICIO + N_SEEDS)
N_GERACOES = 100                # geracoes/iteracoes (valor base)
OBJECTIVE = "Makespan"

print(f"Serao testados {len(SOLVERS_A_TESTAR)} solvers em {len(INSTANCIAS_CONFIG)} instancias")
print(f"com {N_SEEDS} seeds cada = {len(SOLVERS_A_TESTAR) * len(INSTANCIAS_CONFIG) * N_SEEDS} execucoes por experimento")

## 4. Selecao de Instancias

In [ ]:
def selecionar_instancias(data_dir: Path, configs: List[Dict]) -> Dict[str, Path]:
    if not data_dir.exists():
        raise FileNotFoundError(f"Diretorio de instancias nao encontrado: {data_dir}")
    todos = sorted(data_dir.glob("*.txt"))
    if not todos:
        raise FileNotFoundError(f"Nenhum arquivo .txt encontrado em {data_dir}")
    escolhidas = {}
    for cfg in configs:
        candidatos = [f for f in todos if cfg["padrao"] in f.name]
        if not candidatos:
            print(f"Instancia '{cfg['rotulo']}' ({cfg['padrao']}*) nao encontrada — pulando")
            continue
        escolhidas[cfg["rotulo"]] = candidatos[0]
        print(f"{cfg['rotulo']}: {candidatos[0].name}  ({cfg['desc']})")
    if not escolhidas:
        raise RuntimeError("Nenhuma instancia foi encontrada! Verifique DATA_DIR.")
    return escolhidas


instancias = selecionar_instancias(DATA_DIR, INSTANCIAS_CONFIG)

## 5. Funcao auxiliar para montar parametros

Duas versoes:
- **params_originais**: parametros proximos aos defaults que cada aluno usou
  (cada solver tem orcamento computacional diferente — nao e uma comparacao
  justa em termos de numero de avaliacoes, mas mostra o desempenho "como veio")
- **params_padronizados**: todos os solvers com ~5.000 avaliacoes da funcao
  objetivo cada. Isso permite comparar a eficiencia bruta de cada algoritmo
  com o mesmo orcamento computacional.

In [ ]:
# EXPERIMENTO A: parametros originais (cada um com seu custo)
# Motivo: manter os parametros que cada aluno definiu.
# Cada solver tem orcamento diferente (ACO: 3k, BRKGA: 10k avaliacoes).
# Mostra o resultado bruto, sem normalizar por custo.

def params_originais(solver_name: str, n_gen: int) -> dict:
    params = {}
    if solver_name == "ACO":
        params = {
            "n_ants": 30,            # 30 formigas x n_gen iteracoes
            "n_iterations": n_gen,
            "alpha": 0.5,
            "beta": 3.0,
            "rho": 0.3,
            "Q": 200.0,
            "mode": "NPFS",
        }
    elif solver_name.startswith("BRKGA"):
        params = {
            "n_gen": n_gen,           # pop=100 individuos x n_gen geracoes
            "n_elites": 20,
            "n_offsprings": 70,
            "n_mutants": 10,
            "bias": 0.7,
        }
    elif solver_name.startswith("Neuro-BOA"):
        params = {
            "generations": n_gen,    # pop=40 individuos x n_gen geracoes
            "population_size": 40,
            "elite_frac": 0.15,
            "neuro_hidden": 128,
            "neuro_epochs_per_gen": 2,
            "probe_per_generator": 4,
        }
    elif solver_name == "PBIL-Fuzzy":
        params = {
            "max_geracoes": n_gen,   # pop=80 individuos x n_gen geracoes
            "n_pop": 80,
            "sigma_amostragem": 0.10,
            "pct_elite": 0.08,
        }
    return params


# EXPERIMENTO B: parametros padronizados (~5.000 avaliacoes cada)
# Motivo: todos com ~5.000 avaliacoes da funcao objetivo.
# Comparacao mais justa: diferenca no makespan reflete eficiencia
# do algoritmo, nao orcamento maior.
# Calculo: ACO=50x100, BRKGA=100x50, Neuro-BOA=50x100, PBIL=80x62

def params_padronizados(solver_name: str, n_gen: int) -> dict:
    params = {}
    if solver_name == "ACO":
        params = {
            "n_ants": 50,            # 50 x 100 = 5.000 avaliacoes
            "n_iterations": n_gen,   # n_gen = 100
            "alpha": 0.5,
            "beta": 3.0,
            "rho": 0.3,
            "Q": 200.0,
            "mode": "NPFS",
        }
    elif solver_name.startswith("BRKGA"):
        params = {
            "n_gen": 50,             # pop=100 x 50 = 5.000 avaliacoes
            "n_elites": 20,
            "n_offsprings": 70,
            "n_mutants": 10,
            "bias": 0.7,
        }
    elif solver_name.startswith("Neuro-BOA"):
        params = {
            "generations": n_gen,    # pop=50 x 100 = 5.000 avaliacoes
            "population_size": 50,
            "elite_frac": 0.15,
            "neuro_hidden": 128,
            "neuro_epochs_per_gen": 2,
            "probe_per_generator": 4,
        }
    elif solver_name == "PBIL-Fuzzy":
        params = {
            "max_geracoes": 62,      # pop=80 x 62 = 4.960 (~5k)
            "n_pop": 80,
            "sigma_amostragem": 0.10,
            "pct_elite": 0.08,
        }
    return params

## 6. Funcao para rodar um experimento completo

In [ ]:
def rodar_experimento_completo(
    solvers: List[str],
    instancias_disponiveis: Dict[str, Path],
    fn_params,          # funcao que mapeia solver_name -> dict de parametros
    rotulo: str,        # identificador do experimento (ex: "original", "padronizado")
    n_gen: int = 100,
    n_seeds: int = 5,
    seed_inicio: int = 100,
) -> Tuple[pd.DataFrame, dict]:
    """
    Roda todos os solvers em todas as instancias por N seeds.
    Retorna (DataFrame com resultados, dict detalhado).
    """
    resultados_brutos = []
    resultados_detalhados = {}

    total_exec = len(instancias_disponiveis) * len(solvers) * n_seeds
    exec_atual = 0

    print(f"\n{'='*70}")
    print(f"EXPERIMENTO: {rotulo}")
    print(f"{'='*70}")

    for rotulo_inst, inst_path in instancias_disponiveis.items():
        print(f"\n--- Instancia: {rotulo_inst} ({inst_path.name}) ---")

        n_jobs, n_mach, proc_times, due_dates = FlowShopEngine.carregar_instancia_txt(
            str(inst_path)
        )
        engine = FlowShopEngine(n_jobs, n_mach, proc_times, due_dates)
        print(f"  Jobs: {n_jobs}, Maquinas: {n_mach}")

        for solver_name in solvers:
            print(f"  Solver: {solver_name}")

            for seed in range(seed_inicio, seed_inicio + n_seeds):
                exec_atual += 1
                params = fn_params(solver_name, n_gen)

                if exec_atual % 10 == 0:
                    print(f"    [{exec_atual}/{total_exec}] seed {seed}...")

                try:
                    result = rodar_experimento(
                        solver_name=solver_name,
                        engine=engine,
                        seed=seed,
                        objective=OBJECTIVE,
                        params=params,
                        verbose=False,
                    )

                    resultados_brutos.append({
                        "experimento": rotulo,
                        "instancia_rotulo": rotulo_inst,
                        "instancia_arquivo": inst_path.name,
                        "n_jobs": n_jobs,
                        "n_machines": n_mach,
                        "solver": result.solver_name,
                        "versao": result.version,
                        "seed": seed,
                        "makespan": result.best_cost,
                        "tardiness": result.tardiness,
                        "gap_pct": round(result.gap_percent(), 2),
                        "tempo_s": round(result.time_seconds, 2),
                        "n_avaliacoes": result.n_evaluations,
                        "lower_bound": result.lower_bound,
                    })

                    resultados_detalhados[(rotulo, rotulo_inst, solver_name, seed)] = result

                except Exception as e:
                    print(f"    ERRO em {solver_name} seed {seed}: {e}")
                    resultados_brutos.append({
                        "experimento": rotulo,
                        "instancia_rotulo": rotulo_inst,
                        "instancia_arquivo": inst_path.name,
                        "n_jobs": n_jobs,
                        "n_machines": n_mach,
                        "solver": solver_name,
                        "versao": "",
                        "seed": seed,
                        "makespan": float("nan"),
                        "tardiness": float("nan"),
                        "gap_pct": float("nan"),
                        "tempo_s": float("nan"),
                        "n_avaliacoes": 0,
                        "lower_bound": float("nan"),
                    })

    print(f"\nExperimento '{rotulo}' concluido!")
    df = pd.DataFrame(resultados_brutos)
    return df, resultados_detalhados

## 7. Rodar Experimentos

Roda os dois experimentos: A (original) e B (padronizado).
Cada um produz um CSV separado.

In [ ]:
# EXPERIMENTO A: parametros originais (orcamento diferente entre solvers)
df_A, detalhes_A = rodar_experimento_completo(
    solvers=SOLVERS_A_TESTAR,
    instancias_disponiveis=instancias,
    fn_params=params_originais,
    rotulo="original",
    n_gen=N_GERACOES,
    n_seeds=N_SEEDS,
    seed_inicio=SEED_INICIO,
)

# Salva CSV do experimento A
csv_A = RESULTADOS_DIR / "resultados_originais.csv"
df_A.to_csv(csv_A, index=False)
print(f"\nResultados (originais) salvos em: {csv_A}")

In [ ]:
# EXPERIMENTO B: parametros padronizados (~5.000 avaliacoes cada)
df_B, detalhes_B = rodar_experimento_completo(
    solvers=SOLVERS_A_TESTAR,
    instancias_disponiveis=instancias,
    fn_params=params_padronizados,
    rotulo="padronizado",
    n_gen=N_GERACOES,
    n_seeds=N_SEEDS,
    seed_inicio=SEED_INICIO,
)

# Salva CSV do experimento B
csv_B = RESULTADOS_DIR / "resultados_padronizados.csv"
df_B.to_csv(csv_B, index=False)
print(f"\nResultados (padronizados) salvos em: {csv_B}")

## 8. Tabela Comparativa

Gera uma tabela para cada experimento (A e B).

In [ ]:
def gerar_tabela(df: pd.DataFrame) -> pd.DataFrame:
    agrupado = df.groupby(["experimento", "instancia_rotulo", "solver"], dropna=False)
    linhas = []
    for (exp, inst, solver), grupo in agrupado:
        makespans = grupo["makespan"].dropna()
        if len(makespans) == 0:
            continue
        linhas.append({
            "Experimento": exp,
            "Instancia": inst,
            "Solver": solver,
            "Makespan medio": f"{makespans.mean():.1f}",
            "Desvio": f"{makespans.std():.1f}",
            "Melhor": f"{makespans.min():.1f}",
            "Pior": f"{makespans.max():.1f}",
            "Tardiness medio": f"{grupo['tardiness'].mean():.1f}",
            "Tempo (s)": f"{grupo['tempo_s'].mean():.2f}",
            "Avaliacoes": f"{grupo['n_avaliacoes'].mean():.0f}",
            "Gap medio (%)": f"{grupo['gap_pct'].mean():.2f}",
        })
    return pd.DataFrame(linhas)


# Concatena os dois experimentos para tabela unificada
df_completo = pd.concat([df_A, df_B], ignore_index=True)
tabela = gerar_tabela(df_completo)

print("=" * 130)
print("TABELA COMPARATIVA — Makespan por Experimento x Instancia x Solver")
print("=" * 130)
print(tabela.to_string(index=False))

## 9. Graficos

### 9.1 Boxplot — distribuicao do makespan (experimento A vs B, instancia 16x5)

In [ ]:
def plotar_boxplot_comparado(df: pd.DataFrame, instancia_alvo: str = "16x5"):
    """Boxplot lado a lado: experimento A e B para a instancia alvo."""
    df_alvo = df[df["instancia_rotulo"] == instancia_alvo].dropna(subset=["makespan"])
    if df_alvo.empty:
        print(f"Sem dados para instancia {instancia_alvo}")
        return

    exp_rotulos = df_alvo["experimento"].unique()
    fig, axes = plt.subplots(1, len(exp_rotulos), figsize=(7 * len(exp_rotulos), 5))
    if len(exp_rotulos) == 1:
        axes = [axes]

    cores = {"ACO": "#1E2761", "BRKGA": "#2C7A57", "Neuro-BOA": "#E8871E", "PBIL-Fuzzy": "#8C2F39"}

    for ax, exp in zip(axes, sorted(exp_rotulos)):
        df_exp = df_alvo[df_alvo["experimento"] == exp]
        solvers = sorted(df_exp["solver"].unique())
        dados = [df_exp[df_exp["solver"] == s]["makespan"].values for s in solvers]
        bp = ax.boxplot(dados, labels=solvers, patch_artist=True)
        for patch, s in zip(bp["boxes"], solvers):
            patch.set_facecolor(cores.get(s, "gray"))
            patch.set_alpha(0.6)
        ax.set_title(f"Experimento {exp}")
        ax.set_ylabel("Makespan")
        ax.grid(alpha=0.3)

    plt.suptitle(f"Distribuicao do Makespan — Instancia {instancia_alvo}", fontsize=14)
    plt.tight_layout()
    plt.savefig(str(RESULTADOS_DIR / f"boxplot_{instancia_alvo}_comparado.png"), dpi=150)
    plt.show()


plotar_boxplot_comparado(df_completo, "16x5")

### 9.2 Convergencia (experimento A, instancia alvo)

In [ ]:
def plotar_convergencia(detalhes, rotulo_exp="original", instancia_alvo="16x5", seed_plot=None):
    if seed_plot is None:
        seeds = set(s for (e, i, _, s) in detalhes if e == rotulo_exp and i == instancia_alvo)
        seed_plot = sorted(seeds)[len(seeds) // 2]

    fig, ax = plt.subplots(figsize=(11, 5))
    cores = {"ACO": "#1E2761", "BRKGA": "#2C7A57", "Neuro-BOA": "#E8871E", "PBIL-Fuzzy": "#8C2F39"}

    for (e, i, sn, s), result in sorted(detalhes.items()):
        if e != rotulo_exp or i != instancia_alvo or s != seed_plot:
            continue
        if not result.history:
            continue
        ax.plot(result.history,
                label=f"{result.solver_name} ({result.version})",
                color=cores.get(result.solver_name, "gray"), linewidth=1.8)

    ax.set_xlabel("Iteracao / Geracao")
    ax.set_ylabel("Melhor Makespan")
    ax.set_title(f"Convergencia — Experimento {rotulo_exp}, {instancia_alvo} (seed {seed_plot})")
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(str(RESULTADOS_DIR / f"convergencia_{rotulo_exp}_{instancia_alvo}.png"), dpi=150)
    plt.show()


plotar_convergencia(detalhes_A, "original", "16x5")

### 9.3 Gap para Lower Bound (ambos experimentos)

In [ ]:
def plotar_gap_comparado(df: pd.DataFrame, instancia_alvo="16x5"):
    df_alvo = df[df["instancia_rotulo"] == instancia_alvo].dropna(subset=["gap_pct"])
    if df_alvo.empty:
        return

    fig, ax = plt.subplots(figsize=(10, 5))
    cores = {"ACO": "#1E2761", "BRKGA": "#2C7A57", "Neuro-BOA": "#E8871E", "PBIL-Fuzzy": "#8C2F39"}

    exp_rotulos = sorted(df_alvo["experimento"].unique())
    x_offset = 0
    xticks, xticklabels = [], []

    for exp in exp_rotulos:
        df_exp = df_alvo[df_alvo["experimento"] == exp]
        medias = df_exp.groupby("solver")["gap_pct"].agg(["mean", "std"])
        for idx, (solver, row) in enumerate(medias.iterrows()):
            pos = x_offset + idx * 0.3
            ax.bar(pos, row["mean"], yerr=row["std"], width=0.25,
                   color=cores.get(solver, "gray"), alpha=0.7, capsize=3,
                   label=solver if x_offset == 0 else "")
            xticks.append(pos)
            xticklabels.append(f"{exp}\n{solver}")
        x_offset += len(medias) * 0.3 + 0.5

    ax.set_xticks(xticks)
    ax.set_xticklabels(xticklabels, rotation=25, fontsize=8)
    ax.set_ylabel("Gap (%) para Lower Bound")
    ax.set_title(f"Gap para Lower Bound — Instancia {instancia_alvo}")
    ax.legend()
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(str(RESULTADOS_DIR / f"gap_{instancia_alvo}_comparado.png"), dpi=150)
    plt.show()


plotar_gap_comparado(df_completo, "16x5")

### 9.4 Heatmap — tempo de execucao

In [ ]:
def plotar_heatmap_tempo(df: pd.DataFrame):
    pivot = df.pivot_table(
        values="tempo_s", index="instancia_rotulo",
        columns=["experimento", "solver"], aggfunc="mean"
    )

    fig, ax = plt.subplots(figsize=(10, 4))
    im = ax.imshow(pivot.values, cmap="YlOrRd", aspect="auto")
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels([f"{c[0]}\n{c[1]}" for c in pivot.columns], rotation=30, fontsize=7)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    ax.set_title("Tempo Medio de Execucao (s)")

    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            val = pivot.values[i, j]
            if not np.isnan(val):
                cor_texto = "white" if val > pivot.values[~np.isnan(pivot.values)].mean() else "black"
                ax.text(j, i, f"{val:.1f}", ha="center", va="center", color=cor_texto, fontsize=7)

    plt.colorbar(im, ax=ax, label="Tempo (s)")
    plt.tight_layout()
    plt.savefig(str(RESULTADOS_DIR / "heatmap_tempo.png"), dpi=150)
    plt.show()


plotar_heatmap_tempo(df_completo)

## 10. Resumo Final

In [ ]:
print("=" * 70)
print("RESUMO FINAL DOS EXPERIMENTOS")
print("=" * 70)

for exp in sorted(df_completo["experimento"].unique()):
    df_exp = df_completo[df_completo["experimento"] == exp]
    print(f"\n--- Experimento {exp} ---")

    melhores = df_exp.loc[df_exp.groupby("instancia_rotulo")["makespan"].idxmin()]
    print("Melhor solver por instancia:")
    for _, row in melhores.iterrows():
        print(f"  {row['instancia_rotulo']}: {row['solver']} ({row['makespan']:.1f})")

    mais_rapido = df_exp.groupby("solver")["tempo_s"].mean().idxmin()
    tempo = df_exp.groupby("solver")["tempo_s"].mean().min()
    print(f"Solver mais rapido: {mais_rapido} ({tempo:.2f}s medio)")

    menor_gap = df_exp.groupby("solver")["gap_pct"].mean().idxmin()
    gap_v = df_exp.groupby("solver")["gap_pct"].mean().min()
    print(f"Menor gap para LB: {menor_gap} ({gap_v:.2f}% medio)")

print("\nArquivos gerados:")
for f in sorted(RESULTADOS_DIR.glob("*")):
    print(f"  {f.name}")